In [1]:
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from keras.models import Sequential
from keras.models import load_model 
from keras.layers import Dense, Dropout, LSTM,Bidirectional,Embedding,Activation, Flatten, Conv1D,MaxPooling1D,GRU,MultiHeadAttention,BatchNormalization
from tensorflow.keras.optimizers.legacy import Adam
from keras.layers import Dropout,BatchNormalization
from keras import initializers
from keras import regularizers
from keras.layers import BatchNormalization
from keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.utils import class_weight
from tensorflow import keras
from tensorflow.keras.layers import Input, Add, Multiply
from tensorflow.keras.models import Model
import scipy.stats as stats
import numpy as np
import random
import os
from sklearn import feature_selection
import csv

In [16]:
# AR-BB + RGI
timestep=40
data=[['']*timestep]
lr = np.zeros([1, 300])
for i in range(10):
    data+=np.loadtxt(open(r"D:\Pythonnnnnn\python\MyWork\EXP\ComparativeData\SR_Data\SR_RGI-{}.csv".format(i+1), "r"), delimiter=",",dtype='str')[:,1:].tolist()
for i in range(5):
    lr = np.append(lr, (np.loadtxt(open("D:\Pythonnnnnn\python\MyWork\EXP\MyDataset\Dataset_Best\Best-{}.csv".format(i+1), "r"), delimiter=","))[:,1:], axis=0)
data=data[1:]
lr=lr[1:40001,:]

X=[]
for i in range(len(data)):
    X.append([])
    for j in range(timestep):
        if data[i][j]!='' and  data[i][j]!=' ':
            X[i].append(eval(data[i][j]))
            if X[i][j]<0:
                X[i][j]=1
            elif X[i][j]<=7:
                pass
            elif X[i][j]<=14:
                X[i][j]-=3
            else:
                X[i][j]-=9
        else:
            X[i].append(0)
X=np.array(X)

#筛选数据
a=np.sum(lr,axis=1)!=0 
b=np.sum(X,axis=1)!=0
ind= a & b
X=X[ind,:]
lR=lr[ind,:]

LRM=np.zeros([len(lR),10])
for i in range(len(lR)):
    for j in range(10):
        LRM[i,j]=np.mean(lR[i,j*30:(j+1)*30])

y=np.argmin(LRM,axis=1)
label=np.zeros([len(y),301])
label[:,0]=y
label[:,1:]=lR

model = Sequential()
model.add(Embedding(28, 50, input_length=timestep,mask_zero=True))
model.add(Bidirectional(LSTM(50)))
model.add(Dense(10, activation="softmax",
                kernel_initializer=initializers.TruncatedNormal(mean=0.0, stddev=0.1, seed=66),
                kernel_regularizer=regularizers.l2(0.001)))

model = load_model("AR-BB.h5") 

TestY=label[:,0]
TestR=label[:,1:]

prediction = model.predict(X)
y_hat=np.argmax(prediction,axis=1)

accuracy = accuracy_score(TestY, y_hat)
print("准确率:", accuracy)
print(classification_report(TestY,y_hat))

acc=0
for i in range(len(TestY)):
    if TestY[i]==y_hat[i]:
        acc+=1
    elif stats.ranksums(TestR[i,int(y_hat[i]*30):int((y_hat[i]+1)*30)],TestR[i,int(TestY[i]*30):int((TestY[i]+1)*30)])[1] > 0.05:
        # if np.sum(TestR[i,int(y_hat[i]*30):int((y_hat[i]+1)*30)]==10000)<25:
        acc+=1
print("可接受率",acc/len(TestY))



0
1
2
3
4
5
6
7
8
9
1176/1176 [==============================] - 7s 4ms/step
准确率: 0.09094777151366876
              precision    recall  f1-score   support

         0.0       0.50      0.00      0.00      7491
         1.0       0.29      0.04      0.07      8635
         2.0       0.07      0.13      0.09      2343
         3.0       0.08      0.61      0.14      2833
         4.0       0.13      0.01      0.01      8368
         5.0       0.00      0.00      0.00      1469
         6.0       0.12      0.23      0.16      4297
         7.0       0.05      0.02      0.03      1997
         8.0       0.00      0.00      0.00       169
         9.0       0.00      0.00      0.00         2

    accuracy                           0.09     37604
   macro avg       0.12      0.10      0.05     37604
weighted avg       0.22      0.09      0.05     37604


D:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


可接受率 0.4629294755877034


In [3]:
# AR-BB + RGF
timestep=40
data=[['']*timestep]
RGF_label = np.zeros([1])
for i in range(10):
    data+=np.loadtxt(open(r"D:\Pythonnnnnn\python\MyWork\EXP\ComparativeData\SR_Data\SR_RGF-{}.csv".format(i+1), "r"), delimiter=",",dtype='str')[:,1:].tolist()
for i in range(20):
    RGF_label = np.append(RGF_label, np.loadtxt(
                open("D:\Dataset\Mywork\DataAll\RGF\RGF_ELA\lf-{}.csv".format(i + 1), "r"), delimiter=",")[:, 0],
                                  axis=0)
data=data[1:]
RGF_label=RGF_label[1:20001]

X=[]
for i in range(len(data)):
    X.append([])
    for j in range(timestep):
        if data[i][j]!='':
            X[i].append(eval(data[i][j]))
            if X[i][j]<0:
                X[i][j]=1
            elif X[i][j]<=7:
                pass
            elif X[i][j]<=14:
                X[i][j]-=3
            else:
                X[i][j]-=9
        else:
            X[i].append(0)
X=np.array(X)




model = Sequential()
model.add(Embedding(28, 50, input_length=timestep,mask_zero=True))
model.add(Bidirectional(LSTM(50)))
model.add(Dense(10, activation="softmax",
                kernel_initializer=initializers.TruncatedNormal(mean=0.0, stddev=0.1, seed=66),
                kernel_regularizer=regularizers.l2(0.001)))

model = load_model("AR-BB.h5") 

TestY=RGF_label


prediction = model.predict(X)
y_hat=np.argmax(prediction,axis=1)

accuracy = accuracy_score(TestY, y_hat)
print("准确率:", accuracy)
print(classification_report(TestY,y_hat))
# 


625/625 [==============================] - 4s 5ms/step
准确率: 0.51555
              precision    recall  f1-score   support

         0.0       0.60      0.01      0.01       552
         1.0       0.26      0.28      0.27      1333
         2.0       0.56      0.60      0.58      6031
         3.0       0.31      0.51      0.38      1445
         4.0       0.21      0.32      0.25       480
         5.0       0.00      0.00      0.00        26
         6.0       0.67      0.57      0.62      8003
         7.0       0.43      0.44      0.43      1816
         8.0       0.22      0.13      0.17       304
         9.0       0.00      0.00      0.00        10

    accuracy                           0.52     20000
   macro avg       0.33      0.29      0.27     20000
weighted avg       0.54      0.52      0.51     20000


D:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [6]:
# AR-BB + BBOB
timestep=40

data=np.loadtxt(open(r"D:\Pythonnnnnn\python\MyWork\EXP\ComparativeData\SR_Data\SR_BBOB.csv", "r"), delimiter=",",dtype='str')[:,1:].tolist()
lr = np.zeros([1, 300])
for i in range(1,25):
    lr = np.append(lr, np.loadtxt(open("D:\Dataset\Mywork\DataAll\BBOB\\bbob-{}.csv".format(i), "r"),delimiter=",")[:, :300], axis=0)



lr=lr[1:,:]

X=[]
for i in range(len(data)):
    X.append([])
    for j in range(timestep):
        if data[i][j]!='' and  data[i][j]!=' ':
            X[i].append(eval(data[i][j]))
            if X[i][j]<0:
                X[i][j]=1
            elif X[i][j]<=7:
                pass
            elif X[i][j]<=14:
                X[i][j]-=3
            else:
                X[i][j]-=9
        else:
            X[i].append(0)
X=np.array(X)

#筛选数据
a=np.sum(lr,axis=1)!=0 
b=np.sum(X,axis=1)!=0
ind= a & b
X=X[ind,:]
lR=lr[ind,:]

LRM=np.zeros([len(lR),10])
for i in range(len(lR)):
    for j in range(10):
        LRM[i,j]=np.mean(lR[i,j*30:(j+1)*30])

y=np.argmin(LRM,axis=1)
label=np.zeros([len(y),301])
label[:,0]=y
label[:,1:]=lR

model = Sequential()
model.add(Embedding(28, 50, input_length=timestep,mask_zero=True))
model.add(Bidirectional(LSTM(50)))
model.add(Dense(10, activation="softmax",
                kernel_initializer=initializers.TruncatedNormal(mean=0.0, stddev=0.1, seed=66),
                kernel_regularizer=regularizers.l2(0.001)))

model = load_model("AR-BB.h5") 

TestY=label[:,0]
TestR=label[:,1:]

prediction = model.predict(X)
y_hat=np.argmax(prediction,axis=1)

accuracy = accuracy_score(TestY, y_hat)
print("准确率:", accuracy)
print(classification_report(TestY,y_hat))

acc=0
for i in range(len(TestY)):
    if TestY[i]==y_hat[i]:
        acc+=1
    elif stats.ranksums(TestR[i,int(y_hat[i]*30):int((y_hat[i]+1)*30)],TestR[i,int(TestY[i]*30):int((TestY[i]+1)*30)])[1] > 0.05:
        # if np.sum(TestR[i,int(y_hat[i]*30):int((y_hat[i]+1)*30)]==10000)<25:
        acc+=1
print("可接受率",acc/len(TestY))

70/70 [==============================] - 2s 4ms/step
准确率: 0.2675359712230216
              precision    recall  f1-score   support

         1.0       0.00      0.00      0.00         5
         2.0       0.23      0.44      0.30       588
         3.0       0.56      0.31      0.40       854
         4.0       0.00      0.00      0.00        21
         5.0       0.00      0.00      0.00        57
         6.0       0.19      0.15      0.16       472
         7.0       0.01      0.02      0.01        52
         8.0       0.00      0.00      0.00       175

    accuracy                           0.27      2224
   macro avg       0.12      0.11      0.11      2224
weighted avg       0.32      0.27      0.27      2224


D:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


可接受率 0.37140287769784175


In [9]:
# AR-BB + Affine
timestep=40

data=np.loadtxt(open(r"D:\Pythonnnnnn\python\MyWork\EXP\ComparativeData\SR_Data\SR_Affine.csv", "r"), delimiter=",",dtype='str')[:,1:].tolist()
lr= np.loadtxt(open(r"D:\Dataset\Mywork\DataAll\Affine\affine.csv", "r"), delimiter=",")[:,:300]


X=[]
for i in range(len(data)):
    X.append([])
    for j in range(timestep):
        if data[i][j]!='' and  data[i][j]!=' ':
            X[i].append(eval(data[i][j]))
            if X[i][j]<0:
                X[i][j]=1
            elif X[i][j]<=7:
                pass
            elif X[i][j]<=14:
                X[i][j]-=3
            else:
                X[i][j]-=9
        else:
            X[i].append(0)
X=np.array(X)

#筛选数据
a=np.sum(lr,axis=1)!=0 
b=np.sum(X,axis=1)!=0
ind= a & b
X=X[ind,:]
lR=lr[ind,:]

LRM=np.zeros([len(lR),10])
for i in range(len(lR)):
    for j in range(10):
        LRM[i,j]=np.mean(lR[i,j*30:(j+1)*30])

y=np.argmin(LRM,axis=1)
label=np.zeros([len(y),301])
label[:,0]=y
label[:,1:]=lR

model = Sequential()
model.add(Embedding(28, 50, input_length=timestep,mask_zero=True))
model.add(Bidirectional(LSTM(50)))
model.add(Dense(10, activation="softmax",
                kernel_initializer=initializers.TruncatedNormal(mean=0.0, stddev=0.1, seed=66),
                kernel_regularizer=regularizers.l2(0.001)))

model = load_model("AR-BB.h5") 

TestY=label[:,0]
TestR=label[:,1:]

prediction = model.predict(X)
y_hat=np.argmax(prediction,axis=1)

accuracy = accuracy_score(TestY, y_hat)
print("准确率:", accuracy)
print(classification_report(TestY,y_hat))

acc=0
for i in range(len(TestY)):
    if TestY[i]==y_hat[i]:
        acc+=1
    elif stats.ranksums(TestR[i,int(y_hat[i]*30):int((y_hat[i]+1)*30)],TestR[i,int(TestY[i]*30):int((TestY[i]+1)*30)])[1] > 0.05:
        # if np.sum(TestR[i,int(y_hat[i]*30):int((y_hat[i]+1)*30)]==10000)<25:
        acc+=1
print("可接受率",acc/len(TestY))

149/149 [==============================] - 2s 5ms/step
准确率: 0.27578947368421053
              precision    recall  f1-score   support

         1.0       0.04      0.12      0.06        84
         2.0       0.30      0.44      0.35      1372
         3.0       0.54      0.27      0.36      2364
         4.0       0.02      0.01      0.01       136
         5.0       0.00      0.00      0.00        20
         6.0       0.07      0.12      0.09       501
         7.0       0.01      0.02      0.01       149
         8.0       0.00      0.00      0.00       124

    accuracy                           0.28      4750
   macro avg       0.12      0.12      0.11      4750
weighted avg       0.36      0.28      0.29      4750


D:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


可接受率 0.41010526315789475


In [4]:
# AR-BB + MA-BBOB
timestep=40

data=np.loadtxt(open(r"D:\Pythonnnnnn\python\MyWork\EXP\ComparativeData\SR_Data\SR_MA.csv", "r"), delimiter=",",dtype='str')[:,1:].tolist()
lr = np.zeros([1, 300])
for i in range(20):
    lr = np.append(lr, np.loadtxt(open("D:\Pythonnnnnn\python\MyWork\EXP\MA-BBOB\MA-BBOB\\MA-BBOB-{}.csv".format(i), "r"),delimiter=",")[:, 1:301], axis=0)


lr=lr[1:,:]

X=[]
for i in range(len(data)):
    X.append([])
    for j in range(timestep):
        if data[i][j]!='' and  data[i][j]!=' ':
            X[i].append(eval(data[i][j]))
            if X[i][j]<0:
                X[i][j]=1
            elif X[i][j]<=7:
                pass
            elif X[i][j]<=14:
                X[i][j]-=3
            else:
                X[i][j]-=9
        else:
            X[i].append(0)
X=np.array(X)

#筛选数据
a=np.sum(lr,axis=1)!=0 
b=np.sum(X,axis=1)!=0
ind= a & b
X=X[ind,:]
lR=lr[ind,:]

LRM=np.zeros([len(lR),10])
for i in range(len(lR)):
    for j in range(10):
        LRM[i,j]=np.mean(lR[i,j*30:(j+1)*30])

y=np.argmin(LRM,axis=1)
label=np.zeros([len(y),301])
label[:,0]=y
label[:,1:]=lR

model = Sequential()
model.add(Embedding(28, 50, input_length=timestep,mask_zero=True))
model.add(Bidirectional(LSTM(50)))
model.add(Dense(10, activation="softmax",
                kernel_initializer=initializers.TruncatedNormal(mean=0.0, stddev=0.1, seed=66),
                kernel_regularizer=regularizers.l2(0.001)))

model = load_model("AR-BB.h5") 

TestY=label[:,0]
TestR=label[:,1:]

prediction = model.predict(X)
y_hat=np.argmax(prediction,axis=1)

accuracy = accuracy_score(TestY, y_hat)
print("准确率:", accuracy)
print(classification_report(TestY,y_hat))

acc=0
for i in range(len(TestY)):
    if TestY[i]==y_hat[i]:
        acc+=1
    elif stats.ranksums(TestR[i,int(y_hat[i]*30):int((y_hat[i]+1)*30)],TestR[i,int(TestY[i]*30):int((TestY[i]+1)*30)])[1] > 0.05:
        # if np.sum(TestR[i,int(y_hat[i]*30):int((y_hat[i]+1)*30)]==10000)<25:
        acc+=1
print("可接受率",acc/len(TestY))

307/307 [==============================] - 3s 4ms/step
准确率: 0.23929445350734094
              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00        36
         1.0       0.00      0.03      0.00        40
         2.0       0.27      0.59      0.37      2849
         3.0       0.56      0.09      0.16      6467
         4.0       0.00      0.00      0.00        22
         5.0       0.00      0.00      0.00        49
         6.0       0.03      0.14      0.05       302
         7.0       0.01      0.18      0.02        11
         8.0       0.00      0.00      0.00        32

    accuracy                           0.24      9808
   macro avg       0.10      0.11      0.07      9808
weighted avg       0.45      0.24      0.22      9808


D:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


可接受率 0.31290783034257746


In [11]:
# AR-BB + Zigzag
timestep=40

data=np.loadtxt(open(r"D:\Pythonnnnnn\python\MyWork\EXP\ComparativeData\SR_Data\SR_Zigzag.csv", "r"), delimiter=",",dtype='str')[:,1:].tolist()
lr = np.zeros([1, 300])
for i in range(5):
    for j in range(5):
        lr = np.append(lr, np.loadtxt(
            open("D:\Dataset\Mywork\DataAll\Zigzag\\zigzag-{}-{}.csv".format(i, j), "r"), delimiter=",")[:,:300], axis=0)


lr=lr[1:,:]

X=[]
for i in range(len(data)):
    X.append([])
    for j in range(timestep):
        if data[i][j]!='' and  data[i][j]!=' ':
            X[i].append(eval(data[i][j]))
            if X[i][j]<0:
                X[i][j]=1
            elif X[i][j]<=7:
                pass
            elif X[i][j]<=14:
                X[i][j]-=3
            else:
                X[i][j]-=9
        else:
            X[i].append(0)
X=np.array(X)

#筛选数据
a=np.sum(lr,axis=1)!=0 
b=np.sum(X,axis=1)!=0
ind= a & b
X=X[ind,:]
lR=lr[ind,:]

LRM=np.zeros([len(lR),10])
for i in range(len(lR)):
    for j in range(10):
        LRM[i,j]=np.mean(lR[i,j*30:(j+1)*30])

y=np.argmin(LRM,axis=1)
label=np.zeros([len(y),301])
label[:,0]=y
label[:,1:]=lR

model = Sequential()
model.add(Embedding(28, 50, input_length=timestep,mask_zero=True))
model.add(Bidirectional(LSTM(50)))
model.add(Dense(10, activation="softmax",
                kernel_initializer=initializers.TruncatedNormal(mean=0.0, stddev=0.1, seed=66),
                kernel_regularizer=regularizers.l2(0.001)))

model = load_model("AR-BB.h5") 

TestY=label[:,0]
TestR=label[:,1:]

prediction = model.predict(X)
y_hat=np.argmax(prediction,axis=1)

accuracy = accuracy_score(TestY, y_hat)
print("准确率:", accuracy)
print(classification_report(TestY,y_hat))

acc=0
for i in range(len(TestY)):
    if TestY[i]==y_hat[i]:
        acc+=1
    elif stats.ranksums(TestR[i,int(y_hat[i]*30):int((y_hat[i]+1)*30)],TestR[i,int(TestY[i]*30):int((TestY[i]+1)*30)])[1] > 0.05:
        # if np.sum(TestR[i,int(y_hat[i]*30):int((y_hat[i]+1)*30)]==10000)<25:
        acc+=1
print("可接受率",acc/len(TestY))

79/79 [==============================] - 2s 4ms/step
准确率: 0.1432
              precision    recall  f1-score   support

         1.0       0.00      0.00      0.00        12
         2.0       0.95      0.16      0.27      1815
         3.0       0.04      0.73      0.08       102
         4.0       0.00      0.00      0.00         4
         5.0       0.00      0.00      0.00       175
         6.0       0.00      0.00      0.00         0
         7.0       0.00      0.00      0.00         1
         8.0       0.00      0.00      0.00       391

    accuracy                           0.14      2500
   macro avg       0.12      0.11      0.04      2500
weighted avg       0.69      0.14      0.20      2500


D:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Recall 

可接受率 0.1672


In [13]:
# AR-BB + GPB
timestep=40

data=np.loadtxt(open(r"D:\Pythonnnnnn\python\MyWork\EXP\ComparativeData\SR_Data\SR_GPB.csv", "r"), delimiter=",",dtype='str')[:,1:].tolist()

lr = np.zeros([1, 300])

for i in range(5):
    lr = np.append(lr, np.loadtxt(open("D:\Dataset\Mywork\DataAll\GPB\\lfA-{}.csv".format(i), "r"), delimiter=",")[:, :300], axis=0)
for i in range(5):

    lr = np.append(lr, np.loadtxt(open("D:\Dataset\Mywork\DataAll\GPB\\lfB-{}.csv".format(i), "r"),delimiter=",")[:, :300], axis=0)

lr=lr[1:,:]

X=[]
for i in range(len(data)):
    X.append([])
    for j in range(timestep):
        if data[i][j]!='' and  data[i][j]!=' ':
            X[i].append(eval(data[i][j]))
            if X[i][j]<0:
                X[i][j]=1
            elif X[i][j]<=7:
                pass
            elif X[i][j]<=14:
                X[i][j]-=3
            else:
                X[i][j]-=9
        else:
            X[i].append(0)
X=np.array(X)

#筛选数据
a=np.sum(lr,axis=1)!=0 
b=np.sum(X,axis=1)!=0
ind= a & b
X=X[ind,:]
lR=lr[ind,:]

LRM=np.zeros([len(lR),10])
for i in range(len(lR)):
    for j in range(10):
        LRM[i,j]=np.mean(lR[i,j*30:(j+1)*30])

y=np.argmin(LRM,axis=1)
label=np.zeros([len(y),301])
label[:,0]=y
label[:,1:]=lR

model = Sequential()
model.add(Embedding(28, 50, input_length=timestep,mask_zero=True))
model.add(Bidirectional(LSTM(50)))
model.add(Dense(10, activation="softmax",
                kernel_initializer=initializers.TruncatedNormal(mean=0.0, stddev=0.1, seed=66),
                kernel_regularizer=regularizers.l2(0.001)))

model = load_model("AR-BB.h5") 

TestY=label[:,0]
TestR=label[:,1:]

prediction = model.predict(X)
y_hat=np.argmax(prediction,axis=1)

accuracy = accuracy_score(TestY, y_hat)
print("准确率:", accuracy)
print(classification_report(TestY,y_hat))

acc=0
for i in range(len(TestY)):
    if TestY[i]==y_hat[i]:
        acc+=1
    elif stats.ranksums(TestR[i,int(y_hat[i]*30):int((y_hat[i]+1)*30)],TestR[i,int(TestY[i]*30):int((TestY[i]+1)*30)])[1] > 0.05:
        # if np.sum(TestR[i,int(y_hat[i]*30):int((y_hat[i]+1)*30)]==10000)<25:
        acc+=1
print("可接受率",acc/len(TestY))

51/51 [==============================] - 2s 5ms/step
准确率: 0.09975062344139651
              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00       198
         1.0       0.09      0.07      0.08       117
         2.0       0.06      0.25      0.10       103
         3.0       0.10      0.13      0.11       288
         4.0       0.12      0.02      0.04       250
         5.0       0.00      0.00      0.00        42
         6.0       0.17      0.33      0.22       244
         7.0       0.02      0.09      0.03        35
         8.0       0.00      0.00      0.00       315
         9.0       0.00      0.00      0.00        12

    accuracy                           0.10      1604
   macro avg       0.06      0.09      0.06      1604
weighted avg       0.07      0.10      0.07      1604


D:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


可接受率 0.2649625935162095


In [6]:
# AR-BB + MA-BBOB
timestep=40

data=np.loadtxt(open(r"D:\Pythonnnnnn\python\MyWork\EXP\ComparativeData\SR_Data\SR_POP.csv", "r"), delimiter=",",dtype='str')[:,1:].tolist()
lr = np.zeros([1, 300])
for i in range(10):
    lr = np.append(lr, np.loadtxt(open("D:\Pythonnnnnn\python\MyWork\EXP\POP\Dataset_Best\\Best-{}.csv".format(i+1), "r"),delimiter=",")[:, 1:], axis=0)


lr=lr[1:,:]

X=[]
for i in range(len(data)):
    X.append([])
    for j in range(timestep):
        if data[i][j]!='' and  data[i][j]!=' ':
            X[i].append(eval(data[i][j]))
            if X[i][j]<0:
                X[i][j]=1
            elif X[i][j]<=7:
                pass
            elif X[i][j]<=14:
                X[i][j]-=3
            else:
                X[i][j]-=9
        else:
            X[i].append(0)
X=np.array(X)

#筛选数据
a=np.sum(lr,axis=1)!=0 
b=np.sum(X,axis=1)!=0
ind= a & b
X=X[ind,:]
lR=lr[ind,:]

LRM=np.zeros([len(lR),10])
for i in range(len(lR)):
    for j in range(10):
        LRM[i,j]=np.mean(lR[i,j*30:(j+1)*30])

y=np.argmin(LRM,axis=1)
label=np.zeros([len(y),301])
label[:,0]=y
label[:,1:]=lR

model = Sequential()
model.add(Embedding(28, 50, input_length=timestep,mask_zero=True))
model.add(Bidirectional(LSTM(50)))
model.add(Dense(10, activation="softmax",
                kernel_initializer=initializers.TruncatedNormal(mean=0.0, stddev=0.1, seed=66),
                kernel_regularizer=regularizers.l2(0.001)))

model = load_model("AR-BB.h5") 

TestY=label[:,0]
TestR=label[:,1:]

prediction = model.predict(X)
y_hat=np.argmax(prediction,axis=1)

accuracy = accuracy_score(TestY, y_hat)
print("准确率:", accuracy)
print(classification_report(TestY,y_hat))

acc=0
for i in range(len(TestY)):
    if TestY[i]==y_hat[i]:
        acc+=1
    elif stats.ranksums(TestR[i,int(y_hat[i]*30):int((y_hat[i]+1)*30)],TestR[i,int(TestY[i]*30):int((TestY[i]+1)*30)])[1] > 0.05:
        # if np.sum(TestR[i,int(y_hat[i]*30):int((y_hat[i]+1)*30)]==10000)<25:
        acc+=1
print("可接受率",acc/len(TestY))

313/313 [==============================] - 3s 5ms/step
准确率: 0.0038
              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00       682
         1.0       0.29      0.00      0.01      3189
         2.0       0.03      0.01      0.01       423
         3.0       0.00      0.00      0.00         2
         4.0       0.25      0.00      0.00      5270
         5.0       0.00      0.00      0.00       332
         6.0       0.01      0.23      0.03       102
         7.0       0.00      0.00      0.00         0

    accuracy                           0.00     10000
   macro avg       0.07      0.03      0.01     10000
weighted avg       0.23      0.00      0.00     10000


D:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Recall 

可接受率 0.3006


In [10]:
# AR-BB + MA-BBOB
timestep=40

data=np.loadtxt(open(r"D:\Pythonnnnnn\python\MyWork\EXP\ComparativeData\SR_Data\SR_TSFP.csv", "r"), delimiter=",",dtype='str')[:,1:].tolist()
lr = np.zeros([1, 300])
for i in range(20):
    lr = np.append(lr, np.loadtxt(open("D:\Pythonnnnnn\python\MyWork\EXP\TSFP\Dataset_Best\\Best-{}.csv".format(i+1), "r"),delimiter=",")[:, 1:], axis=0)


lr=lr[1:,:]

X=[]
for i in range(len(data)):
    X.append([])
    for j in range(timestep):
        if data[i][j]!='' and  data[i][j]!=' ':
            X[i].append(eval(data[i][j]))
            if X[i][j]<0:
                X[i][j]=1
            elif X[i][j]<=7:
                pass
            elif X[i][j]<=14:
                X[i][j]-=3
            else:
                X[i][j]-=9
        else:
            X[i].append(0)
X=np.array(X)

#筛选数据
a=np.sum(lr,axis=1)!=0 
b=np.sum(X,axis=1)!=0
ind= a & b
X=X[ind,:]
lR=lr[ind,:]

LRM=np.zeros([len(lR),10])
for i in range(len(lR)):
    for j in range(10):
        LRM[i,j]=np.mean(lR[i,j*30:(j+1)*30])

y=np.argmin(LRM,axis=1)
label=np.zeros([len(y),301])
label[:,0]=y
label[:,1:]=lR

model = Sequential()
model.add(Embedding(28, 50, input_length=timestep,mask_zero=True))
model.add(Bidirectional(LSTM(50)))
model.add(Dense(10, activation="softmax",
                kernel_initializer=initializers.TruncatedNormal(mean=0.0, stddev=0.1, seed=66),
                kernel_regularizer=regularizers.l2(0.001)))

model = load_model("AR-BB.h5") 

TestY=label[:,0]
TestR=label[:,1:]

prediction = model.predict(X)
y_hat=np.argmax(prediction,axis=1)

accuracy = accuracy_score(TestY, y_hat)
print("准确率:", accuracy)
print(classification_report(TestY,y_hat))

acc=0
for i in range(len(TestY)):
    if TestY[i]==y_hat[i]:
        acc+=1
    elif stats.ranksums(TestR[i,int(y_hat[i]*30):int((y_hat[i]+1)*30)],TestR[i,int(TestY[i]*30):int((TestY[i]+1)*30)])[1] > 0.05:
        # if np.sum(TestR[i,int(y_hat[i]*30):int((y_hat[i]+1)*30)]==10000)<25:
        acc+=1
print("可接受率",acc/len(TestY))

313/313 [==============================] - 3s 5ms/step
准确率: 0.2299
              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00         0
         1.0       0.00      0.00      0.00         0
         2.0       0.53      0.35      0.42      5729
         3.0       0.45      0.07      0.12      4271
         4.0       0.00      0.00      0.00         0
         6.0       0.00      0.00      0.00         0
         7.0       0.00      0.00      0.00         0
         8.0       0.00      0.00      0.00         0

    accuracy                           0.23     10000
   macro avg       0.12      0.05      0.07     10000
weighted avg       0.50      0.23      0.29     10000


D:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
D:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


可接受率 0.2759
